# ContractorLink: dataset exploration and breach-risk modelling
**Elijah Njugi — final-year project**

This notebook loads the project's published 12,000-row research dataset, checks its composition, and trains the same Random Forest configuration used by ContractorLink. Select **Runtime → Run all** in Google Colab. A free CPU runtime is sufficient; no paid GPU or application credentials are needed.

The research dataset is separate from the application database. It contains anonymized incident categories and simulated records, not ContractorLink account passwords, tickets or uploaded documents. Colab runtime files are temporary: download the exported results before ending the session.

In [ ]:
%pip -q install numpy==2.5.2 pandas==3.0.5 scikit-learn==1.9.0 joblib==1.5.3 matplotlib

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_URL = 'https://raw.githubusercontent.com/ElijahNjugi/ContractorLink/5b965e36e21eac89c23c7368b19ea335ac3daa05/ml/data/hybrid_breach_training.csv'
payload = urlopen(DATA_URL, timeout=60).read()
assert hashlib.sha256(payload).hexdigest() == '5d89292a2a6922feafca44d76e23e3e877edfa122bc6b4130fdbcf5783274747', 'Dataset checksum mismatch'
DATA = Path('hybrid_breach_training.csv')
DATA.write_bytes(payload)
data = pd.read_csv(DATA)
print(f'Loaded {len(data):,} rows and {len(data.columns)} columns')
display(data.head(10))

## Provenance and interpretation
The external portion contains 6,000 sampled initial incident snapshots derived from the [UCI Incident Management Process Enriched Event Log](https://doi.org/10.24432/C57S4H). A further 6,000 records were simulated for ContractorLink scenarios with seed 42. The preparation procedure is available in `ml/prepare_hybrid_dataset.py` in the repository.

**All 1,530 positive breach labels in this prepared dataset come from simulated records.** External labels were taken from the initial snapshot's `made_sla` field, not the final incident outcome. The SLA target duration was assigned from priority, rather than measured from a real contractual deadline. These choices limit the validity of the dataset for predicting real operational breaches.

`record_source` is retained for auditing and excluded from model inputs. Differences in the other fields can still identify the source indirectly. A random holdout from this mixture is an internal experiment, not evidence of real-world or cross-organization performance. No interviews, organization pilot or formal user study is claimed here.

In [ ]:
assert len(data) == 12000
assert set(data['breached'].unique()) == {0, 1}
display(pd.crosstab(data.record_source, data.breached, margins=True))
display(pd.DataFrame({'dtype': data.dtypes.astype(str), 'missing': data.isna().sum(), 'unique': data.nunique()}))
print('Duplicate rows:', int(data.duplicated().sum()))
print('Breach proportion:', round(data.breached.mean(), 4))
pd.crosstab(data.record_source, data.breached).plot.bar(stacked=True, rot=0, figsize=(9, 4))
plt.ylabel('Records'); plt.title('Labels by dataset source'); plt.tight_layout(); plt.show()

## Train and evaluate
Seven categorical and three numeric features enter the pipeline. Missing categorical values use the most frequent value; missing numeric values use the median. One-hot encoding ignores unseen categories. Preprocessing is fitted on the training split only.

The stratified 80/20 split uses seed 42. The forest has 400 trees, a minimum leaf size of 3, and balanced class weights. The evaluation reports a majority-class baseline alongside precision, recall, F1, ROC-AUC and the confusion matrix. Accuracy alone is misleading because most labels are negative.

In [ ]:
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


MODELS = Path("results"); MODELS.mkdir(exist_ok=True)
FEATURES = ["priority", "category", "assignment_group", "location", "contact_type", "impact", "urgency", "reassignment_count", "reopen_count", "sla_target_minutes"]
data = pd.read_csv(DATA); x = data[FEATURES]; y = data["breached"]
# Source logs use textual labels such as "3 - Moderate"; classify by meaning, not inferred dtype.
categorical = ["priority", "category", "assignment_group", "location", "contact_type", "impact", "urgency"]
numeric = ["reassignment_count", "reopen_count", "sla_target_minutes"]
prep = ColumnTransformer([("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical), ("numeric", Pipeline([("impute", SimpleImputer(strategy="median"))]), numeric)])
model = Pipeline([("prep", prep), ("forest", RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight="balanced", random_state=42, n_jobs=-1))])
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.2, stratify=y, random_state=42)
model.fit(x_train, y_train); probabilities = model.predict_proba(x_test)[:, 1]
metrics = {"model_version": "v1", "training_rows": int(len(data)), "breach_rate": float(y.mean()), "roc_auc": float(roc_auc_score(y_test, probabilities)), "classification_report": classification_report(y_test, model.predict(x_test), output_dict=True), "features": FEATURES}
joblib.dump(model, MODELS / "breach_risk_v1.joblib")
(MODELS / "breach_risk_v1_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(json.dumps({"model": "breach_risk_v1.joblib", "roc_auc": metrics["roc_auc"]}, indent=2))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
predictions = model.predict(x_test)
print(classification_report(y_test, predictions, digits=4, zero_division=0))
print('ROC-AUC:', round(roc_auc_score(y_test, probabilities), 6))
print('Always non-breach baseline accuracy:', round(accuracy_score(y_test, np.zeros(len(y_test))), 6))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, display_labels=['No breach', 'Breach'], cmap='Blues')
plt.title('Held-out evaluation (2,400 records)'); plt.tight_layout(); plt.show()
audit = pd.DataFrame({'source': data.loc[x_test.index, 'record_source'], 'actual': y_test, 'predicted': predictions})
display(audit.groupby('source').agg(rows=('actual', 'size'), actual_breaches=('actual', 'sum'), predicted_breaches=('predicted', 'sum')))

## What the results mean
The saved project experiment reported approximately 0.8403 ROC-AUC, 0.6996 accuracy, 0.2813 breach precision and 0.8725 breach recall. Its confusion matrix was `[[1412, 682], [39, 267]]`. The always-negative baseline has 87.25% accuracy but detects no breaches. The model's high recall comes with many false alarms; its score is advisory, not proof that a ticket will miss its SLA.

The prepared CSV also contains 6,227 duplicate rows across its retained columns. Different incidents may share those same values; original incident identifiers were not retained, so independence cannot be established from this CSV. Matching feature profiles can appear in both random splits, adding another limitation to interpretation.

Before operational validation, rebuild the external labels from final outcomes, retain identifiers for grouped splitting, verify the deadline mapping, align training categories with live application fields, and evaluate on independent real ContractorLink outcomes. Repeated incident snapshots must not cross training/test boundaries in future experiments. The current notebook reproduces the disclosed experiment without silently replacing its dataset or claiming improved results.

## Save your work
Use **File → Save a copy in Drive** to keep an editable copy in your Google account. The following optional cell downloads a ZIP containing the research dataset, trained model and metrics. Only load joblib model files from trusted sources.

In [ ]:
import zipfile
with zipfile.ZipFile('ContractorLink_research_results.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(DATA)
    for file in sorted(MODELS.iterdir()):
        archive.write(file)
try:
    from google.colab import files
    files.download('ContractorLink_research_results.zip')
except ImportError:
    print('Saved ContractorLink_research_results.zip in the current directory.')